# GeoNUSAF Baseline — SegNeXt-T (MSCAN-T encoder + LightHam decoder)**Kathmandu Valley satellite land-use segmentation — 6 classes + ignore**Structural twin of `unet.ipynb` and `segformer-b0-new.ipynb`: same 25 sections, same config-dataclasspattern, same asserts, same persistence. Sections 4-11, 13, 15-21, 23-24 are byte-comparable with theB0 notebook so the three files diff cleanly. **Only section 12 is genuinely new.**## What is fixed by protocol (do not change per-model)| Item | Value ||---|---|| Split | 3-fold CV over **408 base tiles** (flip variants travel as one group) || Split modes | `block` (export-sequence proxy) **and** `random` (leakage-gap control) || Loss | effective-number-weighted CE + Lovasz-Softmax, lambda=0.75 || Optimizer | AdamW, differential LR (encoder = head/10) || Schedule | 500-step linear warmup then cosine decay || Stability | `clip_grad_norm_(1.0)`, grad-norm logged every step || Normalization | ImageNet mean/std || Augmentation | rot90 (k in 0..3) + mild photometric jitter, **on-the-fly**. hflip/vflip already baked in and NOT repeated. || Epochs | max 200, early stop on val mIoU, patience 25 || Seed | single seed = 42 || Ignore | index 255, excluded from loss and all metrics || Batch | 16, `GRAD_ACCUM=1` || Shared regularization | `LABEL_SMOOTHING=0.05`, `COS_HORIZON_EPOCHS=120`, `USE_EMA=True` — **matching the B0 run** |## What is model-specific here- **MSCAN-T encoder + LightHamHead decoder**, ~4.23 M params at 6 classes. Written as plain PyTorch  in section 12 — **no mmcv, no mmsegmentation**, nothing to pip-install beyond `ptflops`. The module  and parameter names follow the official repo exactly so the published ImageNet-1K checkpoint loads  without a hand-written conversion table.- **Why this model is in the table.** It is the only entry whose token mixer is neither a plain  convolution stack nor self-attention: MSCA builds its attention map from **multi-scale depthwise  strip convolutions (7, 11, 21)**, and the decoder replaces convolutional context aggregation with a  **low-rank NMF** (the Hamburger module). The strip branches are directionally biased toward  elongated structures, which is the reason to expect something on Road (9.2 px median) and River  (13.2 px) that B0 did not give you. SegNeXt-T is also the published SOTA on iSAID at this size.- **LR: head 6e-4 / backbone 6e-5** — see the box in section 12. This is 10x the B0 notebook's  absolute scale and it is deliberate.## Three things to watch, stated up front1. **The head fuses at 1/8, not 1/4.** `LightHamHead` takes stages 2-4 and aligns them to the   stage-2 grid: logits come out at **64x64** and are upsampled 8x to 512, where B0's come out at   128x128 and are upsampled 4x. A 9 px road is ~1.1 px wide in this logit grid. If SegNeXt   underperforms specifically on Road/River, this is the first suspect — and `FUSE_STAGE1=True`   in the config is a documented one-line ablation that adds stage 1 and fuses at 1/4 instead.   Leave it `False` for the headline row; the published architecture is the baseline.2. **The NMF bases are re-randomised on every forward pass** (`RAND_INIT` is on in the official   implementation). Two evaluations of identical weights therefore differ slightly. This is a   property of the published model, not a bug — do not "fix" it, but expect section 19's final mIoU   to land a few thousandths off the value logged at the best epoch, and say so in the paper.3. **Determinism cost.** Your D6 profiling found `use_deterministic_algorithms(True)` forces a   sort-based kernel for bilinear-upsample backward, costing ~6x. Protocol §6.8 requires it on, so   it is on here — but six SegNeXt runs at that speed sit **at or over your 30 GPU-h weekly quota**.   Section 18 gives both numbers. `DETERMINISTIC=False` is a config field, not a code edit; flipping   it is a protocol change that obliges re-running every baseline.## Session strategy (12 h Kaggle cap)Each **fold is one session**. Full optimizer/scaler/scheduler/EMA state is pushed to Hugging Face as`last.pt`; on restart the notebook resumes automatically. Set `FOLD` and `SPLIT_MODE` in the configcell, run top to bottom, repeat.## Prerequisites- **Internet ON** (Settings -> Internet). Needed for pip, the MSCAN-T ImageNet weights, W&B, HF.- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.- GPU: T4 x1.- **Licensing:** SegNeXt is released under Apache-2.0 by Visual-Attention-Network (Tsinghua). The  model code in section 12 is a derivative work — keep the attribution comment in the cell, and  reproduce the Apache-2.0 notice in whatever code release accompanies the paper.

## 1. EnvironmentNothing model-specific to install. SegNeXt's official implementation depends on `mmcv` +`mmsegmentation 0.24`, which will fight Kaggle's preinstalled torch and cost you a session tountangle; section 12 sidesteps that entirely by defining the ~230 lines of model code inline. Only`ptflops` is missing from the Kaggle image, exactly as in the B0 notebook.

In [ ]:
!pip install -q ptflops==0.7.3 2>&1 | tail -2import os, re, gc, json, time, math, random, warnings, itertools, copy, inspectfrom dataclasses import dataclass, asdict, fieldfrom pathlib import Pathfrom typing import List, Dict, Tuple, Optionalimport numpy as npimport cv2import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoaderwarnings.filterwarnings("ignore")cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUsdef torch_load(path, map_location="cpu"):    """torch>=2.6 defaults weights_only=True, which rejects the argparse.Namespace that    mmseg/openmmlab checkpoints carry in their `meta` block. Try the safe path first and    fall back only when the safe unpickler refuses -- never silently, so a genuinely    corrupt file still raises."""    try:        return torch.load(path, map_location=map_location, weights_only=True)    except Exception as e:        if "WeightsUnpickler" not in str(e) and "weights_only" not in str(e).lower():            raise        print(f"  (weights_only=True refused {Path(path).name}: falling back -- "              f"trusted source, tensors are extracted and the rest discarded)")        return torch.load(path, map_location=map_location, weights_only=False)print("torch", torch.__version__, "| cuda", torch.cuda.is_available())print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration**Set `FOLD` and `SPLIT_MODE` here, then run the whole notebook.** One fold per session.`SPLIT_MODE`:- `"block"` — contiguous runs of export-sequence order as a spatial-block proxy (the reported protocol)- `"random"` — fully random base-tile split (the leakage-gap control)Both still group flip variants, so neither has *duplicate* leakage; the difference between themisolates *spatial* leakage specifically.The architecture block holds the published MSCAN-T settings. `FUSE_STAGE1` and `MD_RAND_INIT` areablation switches, not protocol knobs — leave them at the defaults for the headline row.

In [ ]:
@dataclassclass CFG:    # ---- run selection (change these per session) ----    SPLIT_MODE: str = "block"          # "block" | "random"    FOLD: int = 0                      # 0 | 1 | 2    # ---- data ----    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"    CACHE_DIR: str = "/kaggle/working/cache512"    # If you saved the repacked cache as a Kaggle dataset, point this at it to skip the repack.    PREBUILT_CACHE: Optional[str] = None    IMG_SIZE: int = 512    N_CLASSES: int = 6    IGNORE_INDEX: int = 255    PRELOAD_RAM: bool = True           # ~1.3 GB for the whole 512 cache; kills all disk I/O    # ---- split ----    N_FOLDS: int = 3    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary    # ---- model: MSCAN-T (published tiny config) ----    EMBED_DIMS: tuple = (32, 64, 160, 256)    MLP_RATIOS: tuple = (8, 8, 4, 4)    DEPTHS: tuple = (3, 3, 5, 2)    DROP_PATH_RATE: float = 0.1        # stochastic depth; raise to 0.2 if it still overfits    DROP_RATE: float = 0.0             # MLP dropout inside the encoder (paper: 0)    HEAD_CHANNELS: int = 256    HAM_CHANNELS: int = 256    MD_R: int = 16                     # NMF rank; the tiny config's ham_kwargs value    MD_TRAIN_STEPS: int = 6    MD_EVAL_STEPS: int = 7    MD_RAND_INIT: bool = True          # published behaviour; False = fixed bases (ablation only)    GN_GROUPS: int = 32                # decoder uses GroupNorm, not BatchNorm    CLASSIFIER_DROPOUT: float = 0.1    FUSE_STAGE1: bool = False          # False = stages 2-4 @ 1/8 (published). True = 1/4 ablation.    # ---- pretrained encoder ----    PRETRAINED_URL: str = ("https://download.openmmlab.com/mmsegmentation/v0.5/"                           "pretrain/segnext/mscan_t_20230227-119e8c9f.pth")    PRETRAINED_LOCAL: Optional[str] = None   # path to a manually attached .pth (no-internet fallback)    PRETRAINED_REQUIRED: bool = True         # hard-fail rather than silently train from scratch    PRETRAINED_MIN_COVERAGE: float = 0.98    # ---- optimization ----    BATCH_SIZE: int = 16    GRAD_ACCUM: int = 1                # raise to 2 and halve BATCH_SIZE if you OOM    LR_DECODER: float = 6e-4           # SegNeXt ADE20K recipe: base 6e-5 with head lr_mult=10    LR_ENCODER: float = 6e-5           # 10x differential, per protocol    WEIGHT_DECAY: float = 0.01         # SegNeXt paper value (same as B0; U-Net used 1e-4)    NO_DECAY_ON_NORM_BIAS: bool = True # never decay norm gains, biases, or layer-scale vectors    MAX_EPOCHS: int = 200    PATIENCE: int = 25    MIN_DELTA: float = 0.0            # 0.0 = B0 behaviour. Raise only if you raise it for B0 too.    WARMUP_STEPS: int = 500    CLIP_NORM: float = 1.0    LAMBDA_LOVASZ: float = 0.75    EN_BETA: float = 0.9999            # effective-number beta (Cui et al. 2019)    WEIGHT_CAP: float = 10.0           # cap on normalized class weights    # ---- shared regularization (identical to the B0 run - do not diverge) ----    LABEL_SMOOTHING: float = 0.05    COS_HORIZON_EPOCHS: int = 120    USE_EMA: bool = True    EMA_DECAY: float = 0.999    # ---- augmentation (identical across all baselines) ----    AUG_ROT90: bool = True    AUG_PHOTOMETRIC: bool = True    AUG_BRIGHTNESS: float = 0.10    AUG_CONTRAST: float = 0.10    # ---- overfitting diagnostics (see section 18) ----    TRAIN_EVAL_EVERY: int = 5         # epochs between train-subset evaluations    TRAIN_EVAL_N: int = 400           # images in the fixed un-augmented train probe    # ---- runtime ----    SEED: int = 42    NUM_WORKERS: int = 4    AMP: bool = True    DETERMINISTIC: bool = True         # protocol 6.8. See section 3 for the ~6x cost.    # ---- persistence ----    HF_USER: str = "sugam24"           # <-- B0 lives under Pranilllllll; keep them straight    HF_PREFIX: str = "geonusaf-segnext-t"    PUSH_LAST_EVERY: int = 10    WANDB_PROJECT: str = "geonusaf-baselines"    OUT_DIR: str = "/kaggle/working/out"    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")cfg = CFG()os.makedirs(cfg.OUT_DIR, exist_ok=True)RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)print(f"arch: MSCAN-T dims={cfg.EMBED_DIMS} depths={cfg.DEPTHS} | ham R={cfg.MD_R} "      f"| fuse_stage1={cfg.FUSE_STAGE1}")print(f"opt : lr {cfg.LR_DECODER}/{cfg.LR_ENCODER} wd={cfg.WEIGHT_DECAY} "      f"drop_path={cfg.DROP_PATH_RATE} smooth={cfg.LABEL_SMOOTHING} ema={cfg.USE_EMA}")

## 3. Determinism`use_deterministic_algorithms(warn_only=True)` is deliberate: the decoder upsamples logits 8x withbilinear `interpolate`, whose backward has no deterministic CUDA kernel, and `warn_only=False` wouldhard-crash on the first backward pass.**Read this before you start fold 0.** Your D6 attribution found that this exact setting forces asort-based `indexing_backward_kernel` for that upsample and accounts for ~63% of all CUDA time —SegFormer-B0 went 1967 ms -> 311 ms per step with determinism off. SegNeXt upsamples 8x rather than4x, so it pays the same or slightly more. Protocol §6.8 says determinism on, and all your finishedbaselines paid it, so `DETERMINISTIC=True` here. But be clear-eyed about the arithmetic: sixSegNeXt runs at that speed are **~15-35 GPU-h against a 30 GPU-h weekly quota**. Section 18 has thenumbers both ways. If you decide to flip it, flip it for every baseline and re-run — a table whereone row was trained deterministically and another was not is still a valid table, but a table whereyou *don't know* which is which is not.One extra source of non-determinism this model has and B0 does not: the NMF bases in the decoder aredrawn from `torch.rand` on every forward pass. They are seeded, so a full run replays; but a single`validate()` call on fixed weights is not bit-exact against a second call.

In [ ]:
def set_seed(seed: int):    random.seed(seed); np.random.seed(seed)    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)    os.environ["PYTHONHASHSEED"] = str(seed)    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"    torch.backends.cudnn.deterministic = cfg.DETERMINISTIC    torch.backends.cudnn.benchmark = not cfg.DETERMINISTIC    torch.use_deterministic_algorithms(cfg.DETERMINISTIC, warn_only=True)def worker_init_fn(worker_id):    s = cfg.SEED + worker_id    np.random.seed(s); random.seed(s)set_seed(cfg.SEED)g = torch.Generator(); g.manual_seed(cfg.SEED)print(f"seeded: {cfg.SEED} | deterministic={cfg.DETERMINISTIC} "      f"(cudnn.benchmark={torch.backends.cudnn.benchmark})")

## 4. Filename parsing and base-tile groupingConfirmed convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the`mask` prefix. `N` is ascending **export order with gaps** (mono-class tiles were dropped afterexport), and the three variants of a tile are *not* adjacent in a sorted listing — so grouping isregex-based on `N`, never positional.The assertions here are the guard rail: if a future dataset version renames anything, this cellfails loudly instead of silently splitting a tile across folds.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"MASK_DIR = Path(cfg.DATA_ROOT) / "masks"PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)for f in sorted(os.listdir(IMG_DIR)):    m = PAT.match(f)    assert m, f"unparsed image filename: {f}"    base_id, variant = int(m.group(1)), (m.group(2) or "orig")    mask_name = "mask" + f[len("image"):]    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"    records.append((base_id, variant, f, mask_name))groups: Dict[int, List[tuple]] = {}for r in records:    groups.setdefault(r[0], []).append(r)BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axisassert len(records) == 1224, len(records)assert len(BASE_IDS) == 408, len(BASE_IDS)assert all(len(v) == 3 for v in groups.values())gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")

## 5. Mask colour diagnostic — run this before anything elseYour masks are RGB PNGs; the class table you supplied is in **BGR**. `cv2.imread` returns BGR, sothe keys apply directly — but note that `(128,0,0)` = Residential and `(0,0,128)` = River are exactreverses of each other. **Getting the channel order wrong silently swaps your dominant class withyour rarest one** and every metric downstream would still look plausible.The check below is the safeguard: it dumps every distinct colour actually present and then assertsthat Residential occupies far more pixels than River. It also surfaces stray colours (CVAT polygonedge artifacts, resampling residue) — anything not in the 6-colour table becomes `ignore_index=255`.This also addresses the device-side assert you hit on the earlier SegFormer run: nothing reaches theloss unless it is in `{0..5, 255}`.

In [ ]:
BGR_TO_CLASS = {    (128, 0, 0): 0,    # Residential Area    (0, 128, 0): 1,    # Road    (0, 0, 128): 2,    # River    (0, 128, 128): 3,  # Forest    (128, 128, 0): 4,  # Unused Land    (128, 0, 128): 5,  # Agricultural Area}# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.def pack_bgr(a: np.ndarray) -> np.ndarray:    a = a.astype(np.uint32)    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = ignorefor bgr, idx in BGR_TO_CLASS.items():    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idxdef rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:    return LUT[pack_bgr(bgr_img)]# ---- diagnostic over a random sample of raw full-res masks ----rng = np.random.RandomState(cfg.SEED)sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)   # 7th bucket = ignorefor _, _, _, mname in sample:    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)    for c, n in zip(cols, cnts):        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)    idx = rgbmask_to_index(m)    b = np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)    class_px += bknown = set(BGR_TO_CLASS)print("distinct BGR colours found in sample:")for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):    tag = f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known else "-> IGNORE"    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")tot = class_px.sum()print("\nclass pixel share (sample):")for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.assert class_px[0] > class_px[2] * 3, (    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cacheYour tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs and resizing them onKaggle's 4 vCPUs would leave the T4 idle for most of every epoch.The repack does the resize **once**:- **Images**: `INTER_AREA` (the correct kernel for 4.6x downsampling; bilinear aliases thin roads).- **Masks**: colour -> class index at **full resolution first**, then `INTER_NEAREST` to 512.  Resizing the RGB mask first would interpolate between class colours and manufacture phantom  classes at every polygon edge.Result: ~700 MB instead of 13 GB. Effective GSD after this step is 300 m / 512 px = **0.586 m/px**.**This cache is byte-identical to the one the U-Net and B0 runs used** — if you already saved it as aKaggle dataset, point `PREBUILT_CACHE` at it now and this cell is a no-op (~15-20 min saved).

In [ ]:
from concurrent.futures import ProcessPoolExecutorCACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)MARKER = CACHE / "_complete.json"def _repack_one(rec):    _, _, iname, mname = rec    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)    lab = LUT[pack_bgr(msk)]                                    # full-res -> class index    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)if MARKER.exists():    meta = json.load(open(MARKER))    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")else:    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"    (CACHE / "images").mkdir(parents=True, exist_ok=True)    (CACHE / "labels").mkdir(parents=True, exist_ok=True)    t0 = time.time(); counts = {}    with ProcessPoolExecutor(max_workers=4) as ex:        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):            counts[rec[2]] = b.tolist()            if len(counts) % 200 == 0:                el = time.time() - t0                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  eta {el/len(counts)*(len(records)-len(counts)):.0f}s")    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e9print(f"cache size: {sz:.2f} GB")

## 7. Load the cache into RAM1224 x (512x512x3 + 512x512) uint8 = **~1.3 GB**, comfortably inside Kaggle's ~13 GB. Holding itresident removes disk I/O from the training loop entirely and lets `num_workers` do nothing butaugmentation. Set `PRELOAD_RAM=False` if you ever hit memory pressure.

In [ ]:
RAM = {}if cfg.PRELOAD_RAM:    t0 = time.time()    for i, (_, _, iname, mname) in enumerate(records):        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        RAM[iname] = (img, lab)        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")    # sanity: labels must only ever contain 0..5 or 255    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))    assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"    print("label value check PASSED:", u.tolist())

## 8. Splitters — sequence-block proxy and random control### The proxy, and its honest limitationYou have no surviving grid indices, so contiguous runs of export order stand in for spatial blocks.QGIS Atlas iterates the coverage layer in feature order, which for a rectangular grid is normally**row-major**. That means:- Consecutive ids are neighbours **along the scan axis only**. Adjacency *perpendicular* to it  (the tile directly above/below) is completely uncontrolled — those tiles can land in different folds.- Every row wrap is a spatial discontinuity, so a "block" is really a set of strips.- Dropped mono-class tiles break runs, so id distance overstates ground distance near gaps.Net effect: **this proxy under-estimates true spatial leakage** relative to real geo-blocked CV.Report it as *sequence-block CV (export-order proxy)*, not as spatial CV, and cite the random-splitgap as a lower bound on the leakage effect.### Buffer`BLOCK_BUFFER=2` base tiles adjacent to each held-out block are dropped from **train only** (neverfrom val), so the fold boundary is not two neighbouring tiles sitting on opposite sides of the split.### GroupingBoth splitters operate on the 408 base ids; `expand()` then pulls in all three variants. Thepermutation in `random_split` is seeded from `cfg.SEED` alone, so it is **the same partition theU-Net and B0 used** — the folds compare like with like.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):    n = len(base_ids)    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]    lo, hi = edges[fold], edges[fold + 1]    val = base_ids[lo:hi]    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]    return train, valdef random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):    ids = list(base_ids)    random.Random(seed).shuffle(ids)                 # same permutation for all folds    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]    val = ids[edges[fold]:edges[fold + 1]]    train = ids[:edges[fold]] + ids[edges[fold + 1]:]    return sorted(train), sorted(val)if cfg.SPLIT_MODE == "block":    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)elif cfg.SPLIT_MODE == "random":    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)else:    raise ValueError(cfg.SPLIT_MODE)def expand(ids): return [r for i in ids for r in groups[i]]tr_recs, va_recs = expand(tr_ids), expand(va_ids)assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "      f"({len(BASE_IDS)-len(tr_ids)-len(va_ids)} buffered out)")print(f"  -> {len(tr_recs)} train images / {len(va_recs)} val images")print(f"  val id range: {min(va_ids)}..{max(va_ids)}")

## 9. Effective-number class weights — computed on the train split onlyWeights come from the **training fold alone**. Computing them once over the full dataset would leakval class priors into the loss; small effect, but exactly the kind of thing a Q1 reviewer flags.### Important: beta must scale with the counting unitEffective number (Cui et al. 2019) is `E_c = (1 - beta^n_c) / (1 - beta)`, `w_c = 1/E_c`. The paperapplies it to **per-class image counts** (hundreds to thousands), where `beta=0.9999` is sensiblebecause `1/(1-beta) = 1e4` is the same order as `n_c`.Here `n_c` is a **pixel count** (~1e6 to 1e9). At that scale `beta^n_c` underflows to 0 for *every*class, so `E_c = 1/(1-beta)` for all c and the weights come out **exactly uniform** — i.e. plainunweighted CE, silently. The cell verifies this and auto-selects `beta = 1 - 1/N_total` instead.`EN_BETA_AUTO = True` here, matching `unet.ipynb` and the B0 run. Do not mix this across baselines.

In [ ]:
EN_BETA_AUTO = True     # False = reproduce the legacy (saturated -> uniform) behaviourdef effective_number_weights(recs, cap, n_classes, beta_fixed, auto):    n = np.zeros(n_classes, dtype=np.float64)    for _, _, iname, _ in recs:        n += PIX_COUNTS[iname][:n_classes]    beta = (1.0 - 1.0 / n.sum()) if auto else beta_fixed    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)    w = 1.0 / np.maximum(eff, 1e-12)    w = w / w.mean()    w = np.minimum(w, cap)    return w / w.mean(), n, betaCLASS_W, TRAIN_PX, BETA_USED = effective_number_weights(    tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)print(f"total train pixels: {TRAIN_PX.sum():.3e} | beta used: {BETA_USED:.12f} "      f"| saturation point 1/(1-beta) = {1/(1-BETA_USED):.3e}")print(f"{'class':<14}{'train px %':>12}{'weight':>10}")for i, nm in enumerate(cfg.CLASS_NAMES):    print(f"{nm:<14}{TRAIN_PX[i]/TRAIN_PX.sum()*100:11.2f}%{CLASS_W[i]:10.3f}")print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")if CLASS_W.max() / CLASS_W.min() < 1.05:    print("\nWARNING: weights are effectively uniform -- beta has saturated. "          "This is unweighted CE. Set EN_BETA_AUTO=True unless you are deliberately "          "matching an earlier saturated run.")CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 10. Dataset and DataLoaders**No hflip/vflip** — those are already materialised in the 1224 files. Adding them would produceduplicate views inside a single epoch.On-the-fly augmentation is **rot90 (k in 0..3) + mild brightness/contrast jitter**. Rationale:combined with the baked-in flips this completes the dihedral group D4, which is the natural symmetrygroup for nadir overhead imagery, and it introduces no new interpolation (rot90 is exact — noresampling, so 10-20 px roads stay crisp). Scale jitter is deliberately excluded: it changes effectiveGSD, which would make the thin-structure results harder to interpret.One SegNeXt-specific note on rot90 that is worth a sentence in the paper: MSCA's strip convolutionsare **axis-aligned** (1xk and kx1), so the model is not rotation-equivariant the way an isotropickernel is. rot90 augmentation is therefore doing slightly more work for this architecture than forthe others — it is what teaches the horizontal and vertical branches to cover both road orientations.Keep it identical anyway; that asymmetry is a property of the model under test, not of the protocol.Augmentation is seeded per (epoch, index) so the run is reproducible and workers never collide.Byte-for-byte identical to the U-Net and B0 dataset classes.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)class SatSeg(Dataset):    def __init__(self, recs, train: bool):        self.recs, self.train, self.epoch = recs, train, 0    def __len__(self): return len(self.recs)    def _read(self, iname, mname):        if cfg.PRELOAD_RAM:            img, lab = RAM[iname]            return img.copy(), lab.copy()        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        return img, lab    def __getitem__(self, i):        _, _, iname, mname = self.recs[i]        img, lab = self._read(iname, mname)        if self.train:            rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2**31))            if cfg.AUG_ROT90:                k = rs.randint(4)                if k:                    img = np.ascontiguousarray(np.rot90(img, k))                    lab = np.ascontiguousarray(np.rot90(lab, k))            if cfg.AUG_PHOTOMETRIC:                a = 1.0 + rs.uniform(-cfg.AUG_CONTRAST, cfg.AUG_CONTRAST)        # contrast                b = 255.0 * rs.uniform(-cfg.AUG_BRIGHTNESS, cfg.AUG_BRIGHTNESS)  # brightness                img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)        x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0        x = (x - IMNET_MEAN) / IMNET_STD        x = torch.from_numpy(x.transpose(2, 0, 1))        y = torch.from_numpy(lab.astype(np.int64))        return x, ytrain_ds, val_ds = SatSeg(tr_recs, True), SatSeg(va_recs, False)train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,                      worker_init_fn=worker_init_fn, generator=g)val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,                      worker_init_fn=worker_init_fn)# Fixed, un-augmented subset of TRAIN, used in section 18 for the generalization gap.# Same transform path as val, so train_mIoU and val_mIoU are directly comparable._rs = np.random.RandomState(cfg.SEED)_sub = [tr_recs[i] for i in _rs.choice(len(tr_recs), min(cfg.TRAIN_EVAL_N, len(tr_recs)), replace=False)]traineval_ld = DataLoader(SatSeg(_sub, False), batch_size=cfg.BATCH_SIZE, shuffle=False,                          num_workers=2, pin_memory=True, worker_init_fn=worker_init_fn)print(f"train probe: {len(_sub)} images (un-augmented), evaluated every {cfg.TRAIN_EVAL_EVERY} epochs")xb, yb = next(iter(train_ld))print(xb.shape, xb.dtype, yb.shape, yb.dtype, "| label values:", torch.unique(yb).tolist()[:8])

## 11. Loss — weighted CE + Lovasz-SoftmaxCopied verbatim from the B0 notebook — the loss is protocol, not a per-model choice.Lovasz-Softmax is the convex surrogate of the Jaccard loss (Berman et al. 2018) and is what actuallymoves mIoU on thin classes; weighted CE keeps early optimization stable. `lambda=0.75`.Two implementation details that matter:- The whole loss is computed in **fp32 outside autocast**. Lovasz sorts per-pixel errors and its  gradient is a cumulative-sum difference; in fp16 that accumulates visible error and can produce NaNs.- Ignore pixels are filtered *before* flattening, so 255 never reaches either term.`label_smoothing=0.05` matches the B0 run. Expect a non-zero val-loss floor; compare curve *shape*against B0, not absolute values.

In [ ]:
def lovasz_grad(gt_sorted):    p = len(gt_sorted); gts = gt_sorted.sum()    inter = gts - gt_sorted.float().cumsum(0)    union = gts + (1 - gt_sorted).float().cumsum(0)    jac = 1.0 - inter / union    if p > 1: jac[1:p] = jac[1:p] - jac[0:-1]    return jacdef lovasz_softmax_flat(probas, labels, n_classes):    if probas.numel() == 0: return probas.sum() * 0.0    losses = []    for c in range(n_classes):        fg = (labels == c).float()        if fg.sum() == 0: continue                     # "present" classes only        err = (fg - probas[:, c]).abs()        err_sorted, perm = torch.sort(err, 0, descending=True)        losses.append(torch.dot(err_sorted, lovasz_grad(fg[perm])))    return torch.stack(losses).mean() if losses else probas.sum() * 0.0class ComboLoss(nn.Module):    def __init__(self, weights, lam, ignore, n_classes, smooth=0.0):        super().__init__()        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=ignore, label_smoothing=smooth)        self.lam, self.ignore, self.nc = lam, ignore, n_classes    def forward(self, logits, target):        logits = logits.float()                        # fp32 regardless of autocast        ce = self.ce(logits, target)        B, C, H, W = logits.shape        p = logits.permute(0, 2, 3, 1).reshape(-1, C).softmax(1)        t = target.reshape(-1)        keep = t != self.ignore        lv = lovasz_softmax_flat(p[keep], t[keep], self.nc)        return ce + self.lam * lv, ce.detach(), lv.detach()criterion = ComboLoss(CLASS_W_T, cfg.LAMBDA_LOVASZ, cfg.IGNORE_INDEX,                      cfg.N_CLASSES, cfg.LABEL_SMOOTHING).cuda()# Unweighted, unsmoothed CE -- the clean generalization signal.# The protocol loss above uses class weights fit to THIS FOLD'S TRAIN SPLIT and applies them# to val as well, so train-vs-val on that quantity measures prior shift as much as overfitting.criterion_plain = nn.CrossEntropyLoss(ignore_index=cfg.IGNORE_INDEX).cuda()print("loss ready | lambda =", cfg.LAMBDA_LOVASZ, "| label_smoothing =", cfg.LABEL_SMOOTHING)print("plain CE (unweighted, unsmoothed) logged alongside it as the overfitting signal")

## 12. Model, optimizer, scheduler**This is the only section that differs substantively from the B0 notebook.**### Why the model is written out longhandThe official release depends on `mmcv-full` + `mmsegmentation 0.24.1`, both of which pin CUDA/torchcombinations that Kaggle's image does not match. Rather than lose a session to that, the ~270 linesof actual model code are reproduced here with the mmcv wrappers replaced (`ConvModule` becomes aconv + optional GroupNorm + optional ReLU, `build_norm_layer` becomes the layer). **Module andparameter names follow the official repo exactly**, so the published ImageNet checkpoint loadswithout a conversion table. Source: Visual-Attention-Network/SegNeXt, Apache-2.0.### Why it is a source string rather than ordinary class definitions`SegformerForSemanticSegmentation` can rebuild itself from `transformers`; this class exists only inthis project, so the definition has to travel to Hugging Face alongside the weights or the checkpointis dead on arrival. The cell below therefore holds the code **once**, as a string: it is written to`segnext_model.py`, then `exec`'d into the notebook namespace. What trains and what gets pushed arethe same bytes by construction, with no `inspect.getsource` step that can fail and leave a repo withweights but no architecture. Section 15 pushes the file and then re-queries the repo to confirm itlanded before training starts.### The two ideas being tested| Component | What it replaces | Why it might matter here ||---|---|---|| **MSCA** — DW 5x5, then three parallel DW strip pairs (1x7/7x1, 1x11/11x1, 1x21/21x1), then 1x1, used multiplicatively as an attention map | self-attention (SegFormer) or a plain conv stack (U-Net) | a 21-px strip kernel matches a road's *length* while staying 1 px in its *width* — the receptive-field shape is the point || **LightHamHead** — concat stages 2-4, squeeze to 256, then a **NMF low-rank factorisation** as the context module | ASPP / all-MLP fusion / UPerNet | context modelled as low-rank completion, ~0.32 M params for the whole decoder |### Three implementation choices you should be able to defend1. **The Hamburger runs in fp32.** NMF is a chain of multiplicative updates with a `1e-6` denominator   floor; in fp16 those denominators underflow and the intermediate `bmm` products can exceed 65504.   This is the same fp32-outside-autocast discipline the loss already uses, applied for the same   reason. It costs nothing measurable — the block runs at 64x64 with 256 channels.2. **The NMF iterations run under `no_grad`, with a single differentiable coefficient update at the   end.** That is the published "one-step gradient" of Geng et al. 2021, not a shortcut.3. **`RAND_INIT` stays on.** Bases are redrawn each forward pass. It is stochastic at eval — see   section 3 — but turning it off is an architecture change and would no longer be the published   baseline.### Parameter groups and the LR outlier| Group | LR | Weight decay ||---|---|---|| backbone (`model.backbone`), weights | 6e-5 | 0.01 || backbone, BN/LN gains, biases, `layer_scale_*` | 6e-5 | 0 || decode head, weights | 6e-4 | 0.01 || decode head, GroupNorm gains + biases | 6e-4 | 0 |**`LR_DECODER=6e-4` is 10x the B0 notebook's head LR and that is deliberate.** SegNeXt's ADE20Kconfig is `AdamW(lr=6e-5)` with `custom_keys={'head': lr_mult=10}`, i.e. backbone 6e-5 / head 6e-4 —so this *is* the paper's recipe, read off the config, exactly as UNetFormer's 6e-4 was. The 10xencoder/decoder differential the protocol requires is preserved; only the absolute scale moves.There is also direct evidence from your own runs that the low value would be wrong here. TheLightHam decoder is ~0.32 M randomly-initialised parameters sitting on a pretrained backbone —structurally the same situation as TC-SegFormer v1, where a randomly-initialised head at 6e-5 with a500-step warmup gave mIoU 0.079 at epoch 6 and had to be abandoned. Raising the new-module LR to6e-4 is what fixed that. Do not "harmonise" this down to 6e-5 for tidiness; flag it in the methodssection as a per-architecture value taken from the source paper, which is the line the B0 notebookalready draws.Warmup stays at the protocol's 500 steps rather than SegNeXt's 1500 — note the deviation, it is ashared-protocol field.### What the sanity prints must show- `params: 4.23 M` — hand-derived from the layer table; a different number means the config is wrong.- `pretrained: loaded X/Y backbone tensors (100.0%)` — anything below `PRETRAINED_MIN_COVERAGE`  raises. **If it raises, paste the printed sample of missing/unexpected keys** rather than setting  `PRETRAINED_REQUIRED=False`; a checkpoint-naming drift is a two-minute fix, and training MSCAN-T  from scratch on 272 scenes will not produce a usable row.- `logits after upsample: (1, 6, 512, 512)`.- `head fuses at stride 8 -> logits 64x64` — this is the coarse-head caveat from the title cell,  printed so it ends up in the run log.

In [ ]:
# The model is defined as a source string, written to disk, and then exec'd.# One copy of the code: what runs and what gets pushed to HF cannot diverge, and# there is no inspect.getsource step to fail silently.SEGNEXT_SRC = r'''# ---------------------------------------------------------------------------# SegNeXt-T = MSCAN-T encoder + LightHam decoder.# Ported from https://github.com/Visual-Attention-Network/SegNeXt (Apache-2.0),# mmcv/mmsegmentation dependencies removed. Parameter names match the original# so the published ImageNet-1K MSCAN checkpoints load directly.## This file is the architecture definition that travels with a GeoNUSAF SegNeXt-T# checkpoint. Rebuild with:  SegNeXt(torch.load('best.pt')['arch'])# ---------------------------------------------------------------------------import torchimport torch.nn as nnimport torch.nn.functional as Fclass DropPath(nn.Module):    """Per-sample stochastic depth."""    def __init__(self, p=0.0):        super().__init__()        self.p = p    def forward(self, x):        if self.p == 0.0 or not self.training:            return x        keep = 1.0 - self.p        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)        return x * mask / keepclass ConvModule(nn.Module):    """mmcv.cnn.ConvModule without mmcv: conv -> optional GroupNorm -> optional ReLU.    mmcv's bias='auto' means bias=False whenever a norm follows; reproduced here."""    def __init__(self, cin, cout, k=1, norm=False, act=False, gn_groups=32):        super().__init__()        self.conv = nn.Conv2d(cin, cout, k, bias=not norm)        self.norm = nn.GroupNorm(gn_groups, cout) if norm else None        self.act = nn.ReLU(inplace=True) if act else None    def forward(self, x):        x = self.conv(x)        if self.norm is not None:            x = self.norm(x)        if self.act is not None:            x = self.act(x)        return xclass Mlp(nn.Module):    """1x1 -> depthwise 3x3 -> GELU -> 1x1. Conv-based FFN, no LayerNorm."""    def __init__(self, dim, hidden, drop=0.0):        super().__init__()        self.fc1 = nn.Conv2d(dim, hidden, 1)        self.dwconv = nn.Conv2d(hidden, hidden, 3, 1, 1, bias=True, groups=hidden)        self.act = nn.GELU()        self.fc2 = nn.Conv2d(hidden, dim, 1)        self.drop = nn.Dropout(drop)    def forward(self, x):        x = self.drop(self.act(self.dwconv(self.fc1(x))))        return self.drop(self.fc2(x))class MSCA(nn.Module):    """Multi-Scale Convolutional Attention. Three parallel depthwise strip-conv    branches (7 / 11 / 21) build an attention map that multiplies the input.    The strip shape is what makes this architecture worth testing on 9-13 px    roads and rivers: long along the structure, 1 px across it."""    def __init__(self, dim):        super().__init__()        self.conv0   = nn.Conv2d(dim, dim, 5, padding=2, groups=dim)        self.conv0_1 = nn.Conv2d(dim, dim, (1, 7),  padding=(0, 3),  groups=dim)        self.conv0_2 = nn.Conv2d(dim, dim, (7, 1),  padding=(3, 0),  groups=dim)        self.conv1_1 = nn.Conv2d(dim, dim, (1, 11), padding=(0, 5),  groups=dim)        self.conv1_2 = nn.Conv2d(dim, dim, (11, 1), padding=(5, 0),  groups=dim)        self.conv2_1 = nn.Conv2d(dim, dim, (1, 21), padding=(0, 10), groups=dim)        self.conv2_2 = nn.Conv2d(dim, dim, (21, 1), padding=(10, 0), groups=dim)        self.conv3   = nn.Conv2d(dim, dim, 1)    def forward(self, x):        u = x        a = self.conv0(x)        # RHS is evaluated against the pre-update `a`, matching the reference implementation        a = a + self.conv0_2(self.conv0_1(a)) \              + self.conv1_2(self.conv1_1(a)) \              + self.conv2_2(self.conv2_1(a))        return self.conv3(a) * uclass SpatialAttention(nn.Module):    def __init__(self, dim):        super().__init__()        self.proj_1 = nn.Conv2d(dim, dim, 1)        self.activation = nn.GELU()        self.spatial_gating_unit = MSCA(dim)        self.proj_2 = nn.Conv2d(dim, dim, 1)    def forward(self, x):        s = x        x = self.spatial_gating_unit(self.activation(self.proj_1(x)))        return self.proj_2(x) + sclass MSCABlock(nn.Module):    """BatchNorm (not LayerNorm) + attention + conv-FFN, both with layer scale."""    def __init__(self, dim, mlp_ratio, drop=0.0, drop_path=0.0, ls_init=1e-2):        super().__init__()        self.norm1 = nn.BatchNorm2d(dim)        self.attn = SpatialAttention(dim)        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()        self.norm2 = nn.BatchNorm2d(dim)        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)        self.layer_scale_1 = nn.Parameter(ls_init * torch.ones(dim))        self.layer_scale_2 = nn.Parameter(ls_init * torch.ones(dim))    def forward(self, x):        x = x + self.drop_path(self.layer_scale_1[:, None, None] * self.attn(self.norm1(x)))        x = x + self.drop_path(self.layer_scale_2[:, None, None] * self.mlp(self.norm2(x)))        return xclass StemConv(nn.Module):    """Two stride-2 3x3 convs -> stage 1 at 1/4 resolution."""    def __init__(self, cin, cout):        super().__init__()        self.proj = nn.Sequential(            nn.Conv2d(cin, cout // 2, 3, 2, 1), nn.BatchNorm2d(cout // 2), nn.GELU(),            nn.Conv2d(cout // 2, cout, 3, 2, 1), nn.BatchNorm2d(cout))    def forward(self, x):        return self.proj(x)class OverlapPatchEmbed(nn.Module):    def __init__(self, cin, cout, k=3, stride=2):        super().__init__()        self.proj = nn.Conv2d(cin, cout, k, stride, padding=k // 2)        self.norm = nn.BatchNorm2d(cout)    def forward(self, x):        return self.norm(self.proj(x))class MSCAN(nn.Module):    """Four-stage pyramid at 1/4, 1/8, 1/16, 1/32."""    def __init__(self, embed_dims, mlp_ratios, depths, drop_rate=0.0, drop_path_rate=0.1):        super().__init__()        self.depths = list(depths)        self.embed_dims = list(embed_dims)        dpr = [float(v) for v in torch.linspace(0, drop_path_rate, max(1, sum(depths)))]        cur = 0        for i in range(4):            pe = StemConv(3, embed_dims[0]) if i == 0 \                 else OverlapPatchEmbed(embed_dims[i - 1], embed_dims[i])            blk = nn.ModuleList([MSCABlock(embed_dims[i], mlp_ratios[i], drop_rate, dpr[cur + j])                                 for j in range(depths[i])])            cur += depths[i]            setattr(self, f"patch_embed{i + 1}", pe)            setattr(self, f"block{i + 1}", blk)            setattr(self, f"norm{i + 1}", nn.LayerNorm(embed_dims[i]))    def forward(self, x):        outs = []        for i in range(4):            x = getattr(self, f"patch_embed{i + 1}")(x)            for b in getattr(self, f"block{i + 1}"):                x = b(x)            B, C, H, W = x.shape            # the stage norm is a LayerNorm over channels -> flatten to tokens and back            x = getattr(self, f"norm{i + 1}")(x.flatten(2).transpose(1, 2))            x = x.transpose(1, 2).reshape(B, C, H, W)            outs.append(x)        return outsclass NMF2D(nn.Module):    """The 'ham' in Hamburger: non-negative matrix factorisation as the context    module (Geng et al. 2021). No learnable parameters. The multiplicative-update    iterations run under no_grad; only the final coefficient update is    differentiable, which is the published one-step-gradient scheme."""    def __init__(self, S=1, R=16, train_steps=6, eval_steps=7, rand_init=True, eps=1e-6):        super().__init__()        self.S, self.R = S, R        self.train_steps, self.eval_steps = train_steps, eval_steps        self.rand_init, self.eps = rand_init, eps        self.register_buffer("fixed_bases", torch.empty(0), persistent=False)    def _bases(self, B, D, x):        if self.rand_init or self.fixed_bases.numel() == 0:            b = torch.rand(B * self.S, D, self.R, device=x.device, dtype=x.dtype)            b = F.normalize(b, dim=1)            if not self.rand_init:                self.fixed_bases = b[:self.S].detach()            else:                return b        return self.fixed_bases.repeat(B, 1, 1)    @torch.no_grad()    def _step(self, x, bases, coef):        num = torch.bmm(x.transpose(1, 2), bases)        den = coef.bmm(bases.transpose(1, 2).bmm(bases))        coef = coef * num / (den + self.eps)        num = torch.bmm(x, coef)        den = bases.bmm(coef.transpose(1, 2).bmm(coef))        bases = bases * num / (den + self.eps)        return bases, coef    def forward(self, x):        B, C, H, W = x.shape        D, N = C // self.S, H * W        xf = x.view(B * self.S, D, N)        bases = self._bases(B, D, xf)        with torch.no_grad():            coef = F.softmax(torch.bmm(xf.transpose(1, 2), bases), dim=-1)            for _ in range(self.train_steps if self.training else self.eval_steps):                bases, coef = self._step(xf, bases, coef)        # one differentiable coefficient update -> gradient reaches x, not the bases        num = torch.bmm(xf.transpose(1, 2), bases)        den = coef.bmm(bases.transpose(1, 2).bmm(bases))        coef = coef * num / (den + self.eps)        return torch.bmm(bases, coef.transpose(1, 2)).view(B, C, H, W)class Hamburger(nn.Module):    def __init__(self, ch, R=16, train_steps=6, eval_steps=7, rand_init=True, gn_groups=32):        super().__init__()        self.ham_in  = ConvModule(ch, ch, 1, norm=False, act=False)        self.ham     = NMF2D(R=R, train_steps=train_steps, eval_steps=eval_steps,                             rand_init=rand_init)        self.ham_out = ConvModule(ch, ch, 1, norm=True, act=False, gn_groups=gn_groups)    def forward(self, x):        e = F.relu(self.ham_in(x), inplace=True)        e = self.ham_out(self.ham(e))        return F.relu(x + e, inplace=True)class LightHamHead(nn.Module):    """Concat the selected stages at the coarsest-but-one grid, squeeze, factorise, align, classify."""    def __init__(self, in_channels, channels, ham_channels, n_classes, dropout=0.1,                 R=16, train_steps=6, eval_steps=7, rand_init=True, gn_groups=32):        super().__init__()        self.squeeze   = ConvModule(sum(in_channels), ham_channels, 1, norm=True, act=True,                                    gn_groups=gn_groups)        self.hamburger = Hamburger(ham_channels, R, train_steps, eval_steps, rand_init, gn_groups)        self.align     = ConvModule(ham_channels, channels, 1, norm=True, act=True,                                    gn_groups=gn_groups)        self.dropout   = nn.Dropout2d(dropout)        self.conv_seg  = nn.Conv2d(channels, n_classes, 1)    def forward(self, feats):        size = feats[0].shape[2:]        x = torch.cat([f if f.shape[2:] == size else                       F.interpolate(f, size=size, mode="bilinear", align_corners=False)                       for f in feats], dim=1)        x = self.squeeze(x)        # fp32: NMF's multiplicative updates underflow their 1e-6 floor in fp16 and the        # intermediate bmm products can exceed the fp16 range.        with torch.cuda.amp.autocast(enabled=False):            x = self.hamburger(x.float())        return self.conv_seg(self.dropout(self.align(x)))class SegNeXt(nn.Module):    def __init__(self, arch):        super().__init__()        self.arch = dict(arch)        a = self.arch        self.backbone = MSCAN(a["embed_dims"], a["mlp_ratios"], a["depths"],                              a["drop_rate"], a["drop_path_rate"])        self.in_index = list(a["in_index"])        self.decode_head = LightHamHead(            [a["embed_dims"][i] for i in self.in_index], a["channels"], a["ham_channels"],            a["n_classes"], a["dropout_ratio"], a["md_r"], a["md_train_steps"],            a["md_eval_steps"], a["md_rand_init"], a["gn_groups"])        self._init_head()    def _init_head(self):        for m in self.decode_head.modules():            if isinstance(m, nn.Conv2d):                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")                if m.bias is not None: nn.init.zeros_(m.bias)            elif isinstance(m, nn.GroupNorm):                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)        nn.init.normal_(self.decode_head.conv_seg.weight, std=0.01)        nn.init.zeros_(self.decode_head.conv_seg.bias)    def forward(self, x):        feats = self.backbone(x)        return self.decode_head([feats[i] for i in self.in_index])'''MODEL_SRC_PATH = f"{cfg.OUT_DIR}/segnext_model.py"Path(MODEL_SRC_PATH).write_text(SEGNEXT_SRC)exec(compile(SEGNEXT_SRC, MODEL_SRC_PATH, "exec"), globals())assert Path(MODEL_SRC_PATH).exists() and "SegNeXt" in globals()print(f"model source written and loaded: {MODEL_SRC_PATH} "      f"({len(SEGNEXT_SRC.splitlines())} lines)")

Now instantiate it, load the ImageNet backbone, and build the optimizer.The pretrained loader is deliberately paranoid. It normalises the three naming variants that exist inthe wild (`backbone.` prefix from full segmentation checkpoints, `.dwconv.dwconv.` from the originalrepo where the depthwise conv sat inside a wrapper module, `.projection.` from the mmseg port),drops the ImageNet classifier head, refuses any tensor whose shape disagrees, and then **checkscoverage before returning**. Loading 60% of a backbone and training anyway is the kind of thing thatproduces a plausible-looking bad row in a comparison table.

In [ ]:
ARCH = dict(    embed_dims=list(cfg.EMBED_DIMS), mlp_ratios=list(cfg.MLP_RATIOS), depths=list(cfg.DEPTHS),    drop_rate=cfg.DROP_RATE, drop_path_rate=cfg.DROP_PATH_RATE,    in_index=[0, 1, 2, 3] if cfg.FUSE_STAGE1 else [1, 2, 3],    channels=cfg.HEAD_CHANNELS, ham_channels=cfg.HAM_CHANNELS, n_classes=cfg.N_CLASSES,    dropout_ratio=cfg.CLASSIFIER_DROPOUT, md_r=cfg.MD_R, md_train_steps=cfg.MD_TRAIN_STEPS,    md_eval_steps=cfg.MD_EVAL_STEPS, md_rand_init=cfg.MD_RAND_INIT, gn_groups=cfg.GN_GROUPS)_ALIASES = [(r"^(module\.|backbone\.)+", ""),        # full-seg checkpoints prefix the backbone            (r"\.dwconv\.dwconv\.", ".dwconv."),     # original repo wraps dwconv one level deeper            (r"\.projection\.", ".proj.")]           # mmseg port renamed patch-embed convdef load_mscan_pretrained(model):    if cfg.PRETRAINED_LOCAL and Path(cfg.PRETRAINED_LOCAL).exists():        path, src = cfg.PRETRAINED_LOCAL, "local file"    else:        path, src = f"{cfg.OUT_DIR}/mscan_pretrained.pth", cfg.PRETRAINED_URL        if not Path(path).exists():            torch.hub.download_url_to_file(cfg.PRETRAINED_URL, path)   # needs internet ON    sd = torch_load(path, "cpu")    for k in ("state_dict", "model", "module"):        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict): sd = sd[k]    renamed = {}    for k, v in sd.items():        for pat, rep in _ALIASES: k = re.sub(pat, rep, k)        if k.split(".")[0] in ("head", "fc", "decode_head", "auxiliary_head"): continue        renamed[k] = v    tgt = {k: v for k, v in model.backbone.state_dict().items()           if not k.endswith("num_batches_tracked")}    keep  = {k: v for k, v in renamed.items() if k in tgt and tgt[k].shape == v.shape}    shape = {k: (tuple(v.shape), tuple(tgt[k].shape))             for k, v in renamed.items() if k in tgt and tgt[k].shape != v.shape}    missing = [k for k in tgt if k not in keep]    extra   = [k for k in renamed if k not in tgt]    model.backbone.load_state_dict(keep, strict=False)    cov = len(keep) / max(len(tgt), 1)    print(f"pretrained: loaded {len(keep)}/{len(tgt)} backbone tensors ({cov*100:.1f}%) from {src}")    if shape:  print("  SHAPE MISMATCH  :", list(shape.items())[:5])    if missing: print("  MISSING (target):", missing[:8], f"... +{max(0,len(missing)-8)} more")    if extra:   print("  UNUSED (ckpt)   :", extra[:8], f"... +{max(0,len(extra)-8)} more")    if cov < cfg.PRETRAINED_MIN_COVERAGE and cfg.PRETRAINED_REQUIRED:        raise RuntimeError(            f"only {cov*100:.1f}% of MSCAN-T loaded. Fix the key mapping (paste the MISSING/UNUSED "            f"samples above) rather than training this backbone from scratch.")    return covdef build_model():    return SegNeXt(ARCH)def forward_logits(m, x):    # LightHamHead emits logits at the stride of its first input stage (1/8 by default);    # upsample to input size so loss and metrics stay at native 512, matching every other baseline.    lo = m(x)    return F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)model = build_model()PRETRAIN_COVERAGE = load_mscan_pretrained(model)model = model.cuda()def param_groups(m):    def split(mod):        dec, nod = [], []        for n, p in mod.named_parameters():            if not p.requires_grad: continue            # ndim<=1 catches BN/LN/GN gains, all biases, and both layer_scale vectors            (nod if (p.ndim <= 1 or n.endswith(".bias")) else dec).append(p)        return dec, nod    e_d, e_n = split(m.backbone)    h_d, h_n = split(m.decode_head)    wd = cfg.WEIGHT_DECAY    nwd = 0.0 if cfg.NO_DECAY_ON_NORM_BIAS else wd    return [        {"params": e_d, "lr": cfg.LR_ENCODER, "weight_decay": wd,  "name": "enc_decay"},        {"params": e_n, "lr": cfg.LR_ENCODER, "weight_decay": nwd, "name": "enc_nodecay"},        {"params": h_d, "lr": cfg.LR_DECODER, "weight_decay": wd,  "name": "head_decay"},        {"params": h_n, "lr": cfg.LR_DECODER, "weight_decay": nwd, "name": "head_nodecay"},    ]optimizer = torch.optim.AdamW(param_groups(model), weight_decay=cfg.WEIGHT_DECAY)LR_IDX_ENC, LR_IDX_DEC = 0, 2                  # indices used for loggingsteps_per_epoch = len(train_ld) // cfg.GRAD_ACCUMtotal_steps = steps_per_epoch * cfg.COS_HORIZON_EPOCHS      # cosine horizon, not MAX_EPOCHSdef lr_lambda(step):    if step < cfg.WARMUP_STEPS:        return (step + 1) / cfg.WARMUP_STEPS    prog = (step - cfg.WARMUP_STEPS) / max(1, total_steps - cfg.WARMUP_STEPS)    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)class ModelEMA:    def __init__(self, m, decay):        self.ema = copy.deepcopy(m).eval()        for p in self.ema.parameters(): p.requires_grad_(False)        self.decay = decay    @torch.no_grad()    def update(self, m, step):        # ramp the decay in, otherwise the average stays pinned to the init for thousands of steps        d = min(self.decay, (1.0 + step) / (10.0 + step))        msd = m.state_dict()        for k, v in self.ema.state_dict().items():            # float buffers here include the BatchNorm running stats, which SHOULD be averaged            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)            else: v.copy_(msd[k])ema = ModelEMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else Noneeval_model = ema.ema if cfg.USE_EMA else modeln_par = sum(p.numel() for p in model.parameters())model.eval()with torch.no_grad():    _f = model.backbone(torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))    _head_stride = cfg.IMG_SIZE // _f[ARCH["in_index"][0]].shape[-1]    probe = forward_logits(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda"))model.train()assert probe.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe.shapeprint(f"params: {n_par/1e6:.2f} M   (expected 4.23 M for MSCAN-T + LightHam @ 6 classes)")print(f"stage shapes: {[tuple(t.shape[1:]) for t in _f]}")print(f"head fuses at stride {_head_stride} -> logits "      f"{cfg.IMG_SIZE//_head_stride}x{cfg.IMG_SIZE//_head_stride}, upsampled {_head_stride}x to 512")print(f"logits after upsample: {tuple(probe.shape)}")print(f"steps/epoch: {steps_per_epoch} | cosine horizon: {total_steps} steps "      f"({cfg.COS_HORIZON_EPOCHS} epochs) | EMA: {cfg.USE_EMA}")

## 13. Metrics moduleIdentical to the U-Net and B0 notebooks — the metric code is protocol, so it is copied verbatimrather than rewritten. Everything accumulates into a single 6x6 confusion matrix built on-GPU with`bincount`. Ignore pixels are dropped before accumulation, so 255 never enters any metric.- **IoU / F1 / mIoU / mF1** — macro over the 6 classes- **OA**, **frequency-weighted IoU**- **Cohen's kappa** — from the same matrix- **Producer's accuracy** = recall (row-normalized), **User's accuracy** = precision (column-normalized)- **clDice** (Shit et al. 2021) for **Road** and **River** only — topology-aware, the right lens for  10-20 px linear features, and the metric most likely to separate SegNeXt from B0 if the strip  convolutions are doing what the paper claims. Skeletonization is CPU-bound, so it runs at final  evaluation only.

In [ ]:
from skimage.morphology import skeletonizeclass ConfMat:    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")    def update(self, pred, tgt):        k = tgt != cfg.IGNORE_INDEX        if k.sum() == 0: return        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),                                 minlength=self.n ** 2).reshape(self.n, self.n)    def numpy(self): return self.m.cpu().numpy().astype(np.float64)def metrics_from_cm(cm, names):    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()    iou = tp / np.maximum(tp + fp + fn, 1e-12)    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision    freq = cm.sum(1) / max(tot, 1e-12)    po = tp.sum() / max(tot, 1e-12)    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)    return {        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),        "OA": float(po), "FWIoU": float((freq * iou).sum()),        "kappa": float((po - pe) / max(1 - pe, 1e-12)),        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},        "confusion_matrix_raw": cm.astype(np.int64).tolist(),        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),    }def cldice_pair(pred_bin, gt_bin):    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity    if tprec + tsens == 0: return 0.0    return float(2 * tprec * tsens / (tprec + tsens))

## 14. Efficiency metricsParams, GFLOPs at 512x512, peak training VRAM, wall-clock, and inference FPS, measured exactly as inthe other notebooks so the seven rows are comparable.Three caveats specific to this row:- **`ptflops` hooks `nn.Module`s, so it misses the NMF's functional `bmm` calls entirely.** That is  the same class of undercount as B0's functional attention matmuls, which is convenient — both  transformer-ish rows are undercounted by the same mechanism — but say so in the caption.- **Published FLOP figures are MACs.** The repo quotes SegNeXt-T at "7G" and SegFormer-B0 at "8.4G"  at 512x512; both are MAC counts. This cell reports `2 x MACs`, so the column will read roughly  double the published numbers. That is fine as long as every row uses the same convention — state  which one you used.- **FPS is stochastic** for this model because the NMF bases are redrawn per forward pass; the  iteration count is fixed, so the variance is small, but do not report more than 1 decimal.

In [ ]:
class _FlopsWrapper(nn.Module):    def __init__(self, m): super().__init__(); self.m = m    def forward(self, x): return forward_logits(self.m, x)def efficiency_report(m):    from ptflops import get_model_complexity_info    m.eval()    w = _FlopsWrapper(m)    with torch.cuda.amp.autocast(enabled=False):        macs, params = get_model_complexity_info(            w, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),            as_strings=False, print_per_layer_stat=False, verbose=False)    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):        for _ in range(20): forward_logits(m, x)        torch.cuda.synchronize(); t0 = time.time()        for _ in range(50): forward_logits(m, x)        torch.cuda.synchronize()    return {"params_M": params / 1e6,            "GMACs_512": macs / 1e9,                # published tables quote this figure            "GFLOPs_512": macs * 2 / 1e9,           # ptflops returns MACs; FLOPs = 2 x MACs            "inference_fps_bs1": 50 / (time.time() - t0)}

## 15. Hugging Face persistence and resumeEach fold gets its **own repo** (`geonusaf-segnext-t-{mode}-fold{k}`), matching the U-Net and B0pattern, so any single fold can be pulled independently for inference later and nothing isoverwritten across folds. The aggregation cell in section 21 depends on this layout.Three artifacts per repo:- `best.pt` — weights (EMA weights when `USE_EMA`) + `ARCH` + config + metrics, pushed whenever val  mIoU improves. This is what the inference script downloads.- `last.pt` — full state incl. optimizer/scaler/scheduler/EMA, pushed every `PUSH_LAST_EVERY` epochs.  This is what a new session resumes from.- `segnext_model.py` — the exported architecture source, pushed once. **B0 could rebuild itself from  `transformers`; this model cannot**, so the definition has to travel with the weights or the  checkpoint is unusable in six months. Pushed here, consumed in section 22.At ~4.2 M params both `.pt` files are tens of MB, so upload cost is negligible.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_downloadtry:    from kaggle_secrets import UserSecretsClient    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")except Exception as e:    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)api = HfApi(token=HF_TOKEN)create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)def hf_push(local_path, name, msg):    api.upload_file(path_or_fileobj=local_path, path_in_repo=name,                    repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)def hf_try_download(name):    try: return hf_hub_download(HF_REPO, name, token=HF_TOKEN)    except Exception: return None# the architecture definition must live alongside the weights -- see the note aboveassert Path(MODEL_SRC_PATH).exists(), f"{MODEL_SRC_PATH} missing -- rerun section 12"hf_push(MODEL_SRC_PATH, "segnext_model.py", "architecture source")assert "segnext_model.py" in {f.rfilename for f in api.repo_info(HF_REPO, token=HF_TOKEN).siblings}, \    "segnext_model.py did not land in the repo -- do not start training until it does"print("architecture source verified in", HF_REPO)def push_model_card(best_metrics, epoch):    pc = best_metrics.get("per_class", {})    L = []    L.append("---")    L.append("tags: [semantic-segmentation, remote-sensing, segnext, mscan, geonusaf]")    L.append("license: apache-2.0")    L.append("---")    L.append("")    L.append(f"# GeoNUSAF - SegNeXt-T - {cfg.SPLIT_MODE} split, fold {cfg.FOLD}")    L.append("")    L.append("Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255.")    L.append("")    L.append("| field | value |")    L.append("|---|---|")    L.append("| architecture | MSCAN-T encoder + LightHamHead decoder (SegNeXt, NeurIPS 2022) |")    L.append(f"| encoder init | ImageNet-1K, {PRETRAIN_COVERAGE*100:.1f}% of tensors loaded |")    L.append(f"| params | {n_par/1e6:.2f} M |")    L.append(f"| head fuse stride | {_head_stride} (stages {ARCH['in_index']}) |")    L.append(f"| NMF rank / steps | R={cfg.MD_R}, {cfg.MD_TRAIN_STEPS} train / {cfg.MD_EVAL_STEPS} eval |")    L.append(f"| split mode | {cfg.SPLIT_MODE} |")    L.append(f"| fold | {cfg.FOLD} of {cfg.N_FOLDS} |")    L.append(f"| seed | {cfg.SEED} |")    L.append("| input | 512x512, ImageNet norm, effective GSD 0.586 m/px |")    L.append(f"| classes | {', '.join(cfg.CLASS_NAMES)} |")    L.append(f"| lr (head/encoder) | {cfg.LR_DECODER} / {cfg.LR_ENCODER} |")    L.append(f"| regularization | wd {cfg.WEIGHT_DECAY}, drop_path {cfg.DROP_PATH_RATE}, "             f"smooth {cfg.LABEL_SMOOTHING}, EMA {cfg.USE_EMA} |")    L.append(f"| best epoch | {epoch} |")    L.append(f"| val mIoU | {best_metrics.get('mIoU', float('nan')):.4f} |")    L.append(f"| val mF1 | {best_metrics.get('mF1', float('nan')):.4f} |")    L.append(f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |")    L.append(f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |")    L.append("")    L.append("## Per-class (validation)")    L.append("| class | IoU | F1 |")    L.append("|---|---|---|")    for k, v in pc.items():        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")    L.append("")    L.append("`best.pt` holds `model_state` (EMA weights when EMA is on), `arch` (the dict needed to "             "rebuild the network), the run `cfg`, and `metrics`. Rebuild with `segnext_model.py` "             "from this same repo. Model code derives from Visual-Attention-Network/SegNeXt "             "(Apache-2.0).")    card = "\n".join(L)    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write(card)    hf_push(p, "README.md", f"card epoch {epoch}")print("HF ready:", HF_REPO)

## 16. Weights & BiasesMatches the logging on the other baselines. `resume="allow"` with a deterministic run id means aresumed session continues the same W&B run rather than starting a second one.`overfit_gap = val_loss - train_loss` is logged every epoch, as in the B0 run, so the seven-baselinecomparison of overfitting behaviour stays a one-panel plot.

In [ ]:
import wandbtry:    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))    WANDB = Trueexcept Exception as e:    print("wandb disabled:", e); WANDB = Falseif WANDB:    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,               resume="allow", config={**asdict(cfg), "arch": ARCH, "params_M": n_par / 1e6})

## 17. Train / validate loopsByte-comparable with the B0 notebook. The mechanics worth restating:- `scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`, otherwise you would be clipping  loss-scaled gradients and the 1.0 threshold would mean nothing. `clip_grad_norm_` returns the  pre-clip norm, which is what gets logged — your direct check against the 6e5 explosion from the  earlier inverse-frequency-weighted run.- The EMA updates **after** `scaler.step`, and only on steps the optimizer actually took — a step  skipped by the GradScaler on an inf/NaN check must not advance the average toward unused weights.- Validation runs on the EMA weights at full 512x512 with no TTA.- Early stopping tracks **val mIoU**, not val loss.One thing to watch in the first few hundred steps that is specific to this model: the head LR is6e-4 and the Hamburger's output feeds a residual add. If grad norms sit pinned at the 1.0 clip forthe whole warmup rather than just the first ~100 steps, that is the signal the head LR is too hot —drop to 3e-4 and re-run, and record the change, because it would then apply to every SegNeXt fold.

In [ ]:
def train_one_epoch(epoch):    model.train(); train_ds.epoch = epoch    tot, n, gn_acc = 0.0, 0, []    optimizer.zero_grad(set_to_none=True)    for it, (x, y) in enumerate(train_ld):        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.cuda.amp.autocast(enabled=cfg.AMP):            logits = forward_logits(model, x)        loss, ce, lv = criterion(logits, y)        scaler.scale(loss / cfg.GRAD_ACCUM).backward()        if (it + 1) % cfg.GRAD_ACCUM == 0:            scaler.unscale_(optimizer)                                  # unscale BEFORE clipping            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)            gn_acc.append(float(gn))            prev_scale = scaler.get_scale()            scaler.step(optimizer); scaler.update()            optimizer.zero_grad(set_to_none=True); scheduler.step()            if ema is not None and scaler.get_scale() >= prev_scale:    # skip EMA on skipped steps                ema.update(model, scheduler.last_epoch)        tot += float(loss) * x.size(0); n += x.size(0)        if it % 20 == 0:            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss):.4f} "                  f"(ce {float(ce):.4f} lov {float(lv):.4f}) gnorm {gn_acc[-1] if gn_acc else 0:.3f}",                  flush=True)    return tot / max(n, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))@torch.no_grad()def validate(m=None, loader=None):    m = m if m is not None else eval_model    loader = loader if loader is not None else val_ld    m.eval(); cm = ConfMat(cfg.N_CLASSES); tot, plain, n = 0.0, 0.0, 0    for x, y in loader:        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.cuda.amp.autocast(enabled=cfg.AMP):            logits = forward_logits(m, x)        loss, _, _ = criterion(logits, y)        cm.update(logits.float().argmax(1), y)        tot += float(loss) * x.size(0)        plain += float(criterion_plain(logits.float(), y)) * x.size(0)        n += x.size(0)    return tot / max(n, 1), plain / max(n, 1), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)

## 18. Main loop with automatic resumeOn start this pulls `last.pt` from the fold's HF repo if it exists and continues from that epoch. Sothe answer to "what if it exceeds 12 hours" is: it stops, you start a new session, run the notebooktop to bottom unchanged, and it picks up where it left off.### Expected cost — read this before launching six runs~804 train images, batch 16, **50 steps/epoch**; 408 val images, 26 batches. SegNeXt-T is 7 GMACs vsB0's 8.4, but that understates its cost on a T4: the MSCA branches are ~50 depthwise convolutionsper forward pass, several with 1x21 / 21x1 kernels, and depthwise convs are bandwidth-bound ratherthan FLOP-bound. Expect it to land near B0's step time or slightly above, not below.| | per step | per epoch (incl. val) | per fold (60-140 ep) | six runs ||---|---|---|---|---|| `DETERMINISTIC=True` (protocol) | ~2.0-2.8 s | ~2.5-3.5 min | **2.5-6 h** | **15-35 GPU-h** || `DETERMINISTIC=False` | ~0.45-0.7 s | ~50-70 s | ~1-2 h | ~6-12 h |**The top row is the problem.** Three block folds plus three random folds under the protocol settingsit at or over your 30 GPU-h weekly quota, and each individual fold can exceed the 12 h session cap,so plan on at least one resume for the slower folds. Concretely: run the three **block** folds first(they are the reported protocol), then the three **random** control folds in the following week ifthe quota is tight. The leakage-gap number in section 21 needs all six, but the block rows alone arewhat goes in the main table.**VRAM.** Stage 1 runs at 128x128 with `mlp_ratio=8`, so the widest activation is 16x256x128x128 —larger than anything in B0. Expect **7-10 GB of 16 GB** at batch 16 with AMP. That should fit, but itis the tightest of your lightweight models; if it OOMs, `BATCH_SIZE=8, GRAD_ACCUM=2` keeps theeffective batch at 16 and the protocol unchanged. Do not drop below effective batch 16 — MSCAN usesBatchNorm throughout, and unlike B0 (LayerNorm/GroupNorm) it is genuinely batch-size sensitive.**First-fold checks.** By epoch ~10, val mIoU should be clearly above 0.2. If it is still near 0.08the way TC-SegFormer v1 was, the head LR or the pretrained load is wrong — stop and check section12's coverage print rather than letting it run.### Reading the loss curves — why `val_loss` rising is not the overfitting testEvery model in this benchmark produces the same fold-0 signature: val loss bottoms in the first ~10epochs and then climbs for the rest of the run. U-Net (bottom ~e8-10 at 1.3-1.5, climbing to 2.5-2.8),SegFormer-B0 (1.87 → 2.19, stopped e38), TC-SegFormer (CE bottom ~e24 at 0.392, climbing to 0.576).It is a property of this fold and this loss, not of any one architecture.Two mechanisms produce it, and neither is memorization:1. **The class weights are fit to the fold's train split and then applied to val.** On a block split   the val fold's class priors differ substantially, so the weighted val loss carries a constant   offset that has nothing to do with generalization. `overfit_gap = val_loss − train_loss` is   therefore not a clean measure — it is still logged for continuity with the other notebooks, but do   not draw conclusions from it.2. **The model gets more confident faster than it gets more correct.** Cross-entropy punishes   confident errors quadratically harder than hesitant ones, so a model whose argmax is steadily   improving can still have a rising CE. This is exactly why the protocol early-stops on **val mIoU**,   not val loss — stopping at the val-loss minimum here would hand you the epoch-8 model.**So this notebook logs two honest signals instead.** Every `TRAIN_EVAL_EVERY` epochs it evaluates afixed, **un-augmented** subset of the training split through the same path as validation, and prints:| signal | what it means ||---|---|| `gap_mIoU` = train mIoU − val mIoU | the real generalization gap. Widening while val mIoU flattens = genuine overfitting. || `gap_ce` = val plain CE − train plain CE | same comparison on **unweighted, unsmoothed** CE, so the class-weight offset is gone || `val_ce_plain` | val CE with no weighting and no smoothing — comparable across folds |**One number from the fold-0 curves is already informative: train loss plateaued near 1.05.** U-Netdrove its train loss to ~0.05 — that is memorization. A model sitting at 1.05 on aweighted + smoothed + Lovász objective is not memorizing the training set, which makes prior shiftthe far more likely explanation for the rising val curve. Confirm it with `gap_mIoU` on the next runbefore changing any hyperparameter.### If `gap_mIoU` does confirm overfittingAct in this order, and apply whatever you change to **all six SegNeXt runs**:1. `DROP_PATH_RATE` 0.1 → 0.2. Deliberately left at **0.1 here**, because that is what SegFormer-B0   used — raising it for SegNeXt alone is unequal tuning unless you disclose it, and a reviewer will   ask. If you raise it, raise it as a disclosed per-architecture choice or add it as an extra row.2. `DROP_RATE` 0 → 0.1 (encoder MLP dropout). Departs from the paper config; ablation row, not the   headline.3. Nothing else. Do not touch augmentation, the split, or the loss — those are shared across all five   models, and changing them here invalidates the comparison rather than improving it.

In [ ]:
ck = hf_try_download("last.pt")start_epoch, best_miou, best_epoch, bad, history = 0, -1.0, -1, 0, []if ck:    st = torch_load(ck, "cuda")    model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optim_state"])    scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])    if ema is not None and st.get("ema_state") is not None:        ema.ema.load_state_dict(st["ema_state"])    start_epoch = st["epoch"] + 1; best_miou = st["best_miou"]; best_epoch = st["best_epoch"]    bad = st["bad"]; history = st["history"]    print(f"RESUMED from epoch {st['epoch']} (best mIoU {best_miou:.4f} @ e{best_epoch})")else:    print("fresh start")torch.cuda.reset_peak_memory_stats()t_start, best_metrics = time.time(), {}for epoch in range(start_epoch, cfg.MAX_EPOCHS):    te = time.time()    tr_loss, gn_mean, gn_max = train_one_epoch(epoch)    va_loss, va_ce, m = validate()    peak_gb = torch.cuda.max_memory_allocated() / 1e9    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,           "val_ce_plain": va_ce,           "overfit_gap": va_loss - tr_loss, "val_mIoU": m["mIoU"],           "val_mF1": m["mF1"], "val_OA": m["OA"], "val_kappa": m["kappa"],           "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,           "lr_enc": scheduler.get_last_lr()[LR_IDX_ENC],           "lr_dec": scheduler.get_last_lr()[LR_IDX_DEC], "peak_vram_gb": peak_gb,           "epoch_sec": time.time() - te,           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}    # The real generalization gap: same weights, same transform, train vs val.    # train_loss vs val_loss cannot answer this -- the class weights are fit to the    # train split and the training pass sees augmented images.    if epoch % cfg.TRAIN_EVAL_EVERY == 0:        _, tr_ce, tm = validate(loader=traineval_ld)        row["train_mIoU"] = tm["mIoU"]; row["train_ce_plain"] = tr_ce        row["gap_mIoU"] = tm["mIoU"] - m["mIoU"]; row["gap_ce"] = va_ce - tr_ce        print(f"       probe: train mIoU {tm['mIoU']:.4f} vs val {m['mIoU']:.4f} "              f"(gap {row['gap_mIoU']:+.4f}) | plain CE train {tr_ce:.3f} vs val {va_ce:.3f} "              f"(gap {row['gap_ce']:+.3f})")    history.append(row)    if WANDB: wandb.log(row, step=epoch)    print(f"[e{epoch}] train {tr_loss:.4f} | val {va_loss:.4f} | gap {row['overfit_gap']:+.3f} | "          f"mIoU {m['mIoU']:.4f} | plainCE {va_ce:.3f} | gnorm mu{gn_mean:.2f}/max{gn_max:.2f} | "          f"{row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB")    if m["mIoU"] > best_miou + cfg.MIN_DELTA:        best_miou, best_epoch, bad, best_metrics = m["mIoU"], epoch, 0, m        p = f"{cfg.OUT_DIR}/best.pt"        torch.save({"model_state": eval_model.state_dict(),      # EMA weights when EMA is on                    "arch": ARCH,                                # everything needed to rebuild                    "cfg": asdict(cfg), "metrics": m, "epoch": epoch}, p)        hf_push(p, "best.pt", f"best mIoU {best_miou:.4f} @ e{epoch}")        push_model_card(best_metrics, epoch)        print(f"  -> new best {best_miou:.4f}, pushed")    else:        bad += 1    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):        p = f"{cfg.OUT_DIR}/last.pt"        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),                    "sched_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),                    "ema_state": (ema.ema.state_dict() if ema is not None else None),                    "epoch": epoch, "best_miou": best_miou, "best_epoch": best_epoch,                    "bad": bad, "history": history}, p)        hf_push(p, "last.pt", f"state e{epoch}")    if bad >= cfg.PATIENCE:        print(f"EARLY STOP at epoch {epoch} (best {best_miou:.4f} @ e{best_epoch})"); breakwall_h = (time.time() - t_start) / 3600print(f"training done | best mIoU {best_miou:.4f} @ epoch {best_epoch} | this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpointReloads `best.pt` (EMA weights) into a fresh model and computes the **full** metric suite on theheld-out fold, including clDice for Road and River and the efficiency block. clDice is averaged overimages in which the class is present in GT or prediction; images where the class is absent from bothare skipped rather than scored as 1.0, which would inflate the number badly given how sparse River is.**Expect a small disagreement with the logged best-epoch mIoU** — usually a few thousandths. The NMFbases are redrawn on every forward pass, so this is a second sample of a slightly stochasticevaluation, not a bug and not a checkpointing error. Report *this* number (it is the one computedalongside clDice and the efficiency block) and note the mechanism in the methods section. If the gapis larger than ~0.005, that is worth investigating.

In [ ]:
_ck = torch_load(f"{cfg.OUT_DIR}/best.pt", "cuda")best_model = SegNeXt(_ck["arch"]).cuda()best_model.load_state_dict(_ck["model_state"])best_model.eval()cm = ConfMat(cfg.N_CLASSES)cld = {1: [], 2: []}                                   # Road, Riverwith torch.no_grad():    for x, y in val_ld:        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.cuda.amp.autocast(enabled=cfg.AMP):            logits = forward_logits(best_model, x)        pred = logits.float().argmax(1)        cm.update(pred, y)        pn, yn = pred.cpu().numpy(), y.cpu().numpy()        valid = yn != cfg.IGNORE_INDEX        for b in range(pn.shape[0]):            for c in cld:                pb = (pn[b] == c) & valid[b]                gb = (yn[b] == c) & valid[b]                v = cldice_pair(pb, gb)                if v is not None: cld[c].append(v)final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}final.update(efficiency_report(best_model))final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)final["train_wall_clock_h_this_session"] = wall_hfinal["best_epoch"] = best_epochfinal["logged_best_mIoU"] = best_miou          # for the NMF re-sampling comparison belowfinal["pretrain_coverage"] = PRETRAIN_COVERAGEfinal["head_fuse_stride"] = _head_stridefinal["run"] = {"model": "SegNeXt-T", "split_mode": cfg.SPLIT_MODE,                "fold": cfg.FOLD, "seed": cfg.SEED}print(json.dumps({k: v for k, v in final.items()                  if k not in ("confusion_matrix_raw", "confusion_matrix_norm", "per_class")}, indent=2))print("\nper class:")for k, v in final["per_class"].items():    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}")d = final["mIoU"] - best_miouprint(f"\nre-evaluation delta vs logged best: {d:+.5f}  "      f"({'expected NMF resampling noise' if abs(d) < 5e-3 else 'LARGER THAN EXPECTED -- investigate'})")

## 20. Persist resultsWrites `results_{mode}_fold{k}.json` (full, incl. confusion matrices), a flat `.csv` row for thecomparison table, and `history.csv` for the curves — then pushes all three to the fold's HF repo sothey survive the session. The flat row carries the regularization settings **and** the twoarchitecture switches (`fuse_stage1`, `md_rand_init`) as columns, so an ablation run can never besilently averaged together with the headline run.

In [ ]:
import csvtag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"jp = f"{cfg.OUT_DIR}/results_{tag}.json"json.dump(final, open(jp, "w"), indent=2)flat = {"model": "SegNeXt-T", "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,        "mIoU": final["mIoU"], "mF1": final["mF1"], "OA": final["OA"],        "FWIoU": final["FWIoU"], "kappa": final["kappa"],        "params_M": final["params_M"], "GMACs_512": final["GMACs_512"],        "GFLOPs_512": final["GFLOPs_512"],        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],        "best_epoch": final["best_epoch"],        "lr_dec": cfg.LR_DECODER, "lr_enc": cfg.LR_ENCODER, "weight_decay": cfg.WEIGHT_DECAY,        "label_smoothing": cfg.LABEL_SMOOTHING, "cos_horizon": cfg.COS_HORIZON_EPOCHS,        "ema": cfg.USE_EMA, "drop_path": cfg.DROP_PATH_RATE,        "fuse_stage1": cfg.FUSE_STAGE1, "md_r": cfg.MD_R, "md_rand_init": cfg.MD_RAND_INIT,        "deterministic": cfg.DETERMINISTIC, "pretrain_coverage": final["pretrain_coverage"],        "min_delta": cfg.MIN_DELTA}_h = [r for r in history if r["epoch"] == best_epoch and "gap_mIoU" in r]if _h:    flat["train_mIoU_at_best"] = _h[0]["train_mIoU"]    flat["gap_mIoU_at_best"] = _h[0]["gap_mIoU"]    flat["gap_ce_at_best"] = _h[0]["gap_ce"]for k, v in final["per_class"].items():    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]for k, v in final["clDice"].items():    flat[f"clDice_{k}"] = vcp = f"{cfg.OUT_DIR}/results_{tag}.csv"with open(cp, "w", newline="") as f:    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)hp = f"{cfg.OUT_DIR}/history_{tag}.csv"with open(hp, "w", newline="") as f:    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv")]:    hf_push(p, n, "results")if WANDB: wandb.finish()print("results pushed to", HF_REPO)

## 21. Aggregation across folds — run after all folds finishPulls each fold's CSV from its own HF repo and reports mean +/- std per split mode. The**random-mode mIoU minus block-mode mIoU is the leakage gap**. Expect random to score higher; if thegap is near zero, that is itself a finding — it would suggest the export-order proxy is not capturingmuch spatial structure, consistent with the row-major limitation in section 8.The second block prints where val mIoU actually peaked versus where val loss bottomed, per fold — thediagnostic that tells you whether the shared regularization is load-bearing or cosmetic.The third block is new and specific to this notebook: a **direct thin-structure comparison againstSegFormer-B0**, pulled from the B0 repos. The whole reason SegNeXt-T is in the table is the claimthat multi-scale strip convolutions help on elongated features, so Road/River IoU and clDice are thecolumns where that claim either shows up or does not. Set `B0_USER`/`B0_PREFIX` to wherever the B0folds landed; if they are missing the block just prints a note and moves on.

In [ ]:
import pandas as pdrows = []for mode in ["block", "random"]:    for k in range(cfg.N_FOLDS):        repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{mode}-fold{k}"        try:            rows.append(pd.read_csv(hf_hub_download(repo, f"results_{mode}_fold{k}.csv", token=HF_TOKEN)))        except Exception as e:            print(f"  missing {mode} fold {k}: {type(e).__name__}")if rows:    df = pd.concat(rows, ignore_index=True)    df.to_csv(f"{cfg.OUT_DIR}/aggregate_segnext_t.csv", index=False)    num = df.select_dtypes("number").columns    agg = df.groupby("split_mode")[num].agg(["mean", "std"])    print(agg[[c for c in [("mIoU","mean"),("mIoU","std"),("mF1","mean"),("kappa","mean")] if c in agg]])    if {"block", "random"} <= set(df.split_mode):        b = df[df.split_mode == "block"].mIoU.mean(); r = df[df.split_mode == "random"].mIoU.mean()        print(f"\nLEAKAGE GAP (random - block) mIoU: {r-b:+.4f}  [block {b:.4f} | random {r:.4f}]")# ---- overfitting diagnostic: where did each curve actually turn? ----print("\nval-loss minimum vs val-mIoU maximum per fold")for mode in ["block", "random"]:    for k in range(cfg.N_FOLDS):        repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{mode}-fold{k}"        try:            h = pd.read_csv(hf_hub_download(repo, f"history_{mode}_fold{k}.csv", token=HF_TOKEN))        except Exception:            continue        el, em = int(h.val_loss.idxmin()), int(h.val_mIoU.idxmax())        print(f"  {mode} f{k}: val_loss min @ e{el} ({h.val_loss.min():.3f}) | "              f"val_mIoU max @ e{em} ({h.val_mIoU.max():.4f}) | "              f"mIoU at loss-min e{el} = {h.val_mIoU[el]:.4f}")        if "gap_mIoU" in h:            gp = h.dropna(subset=["gap_mIoU"])            print(f"        train-vs-val mIoU gap: {gp.gap_mIoU.iloc[0]:+.3f} (first probe) -> "                  f"{gp.gap_mIoU.iloc[-1]:+.3f} (last) | plain-CE gap "                  f"{gp.gap_ce.iloc[0]:+.3f} -> {gp.gap_ce.iloc[-1]:+.3f}")# ---- the thin-structure question this model was added to answer ----B0_USER   = "Pranilllllll"                          # <-- where the B0 folds liveB0_PREFIX = "segformer-b0-kaggle-ktm-dataset"b0 = []for k in range(cfg.N_FOLDS):    try:        b0.append(pd.read_csv(hf_hub_download(f"{B0_USER}/{B0_PREFIX}-block-fold{k}",                                              f"results_block_fold{k}.csv", token=HF_TOKEN)))    except Exception:        passif rows and b0:    a = df[df.split_mode == "block"]; b = pd.concat(b0, ignore_index=True)    print("\nblock-split thin-structure comparison (mean over folds)")    print(f"  {'metric':<20}{'SegNeXt-T':>12}{'SegFormer-B0':>14}{'delta':>10}")    for col in ["IoU_Road", "IoU_River", "clDice_Road", "clDice_River", "mIoU", "params_M"]:        if col in a and col in b:            x, y = a[col].mean(), b[col].mean()            print(f"  {col:<20}{x:12.4f}{y:14.4f}{x-y:+10.4f}")else:    print("\n(B0 results not found -- set B0_USER/B0_PREFIX to run the thin-structure comparison)")

## 22. Standalone inference from Hugging FaceFully self-contained: point `INFER_REPO` at any fold's repo and it downloads **both** the weights and`segnext_model.py`, then rebuilds the network from the `arch` dict embedded in `best.pt`. Nothingabove this cell is required — paste it into a fresh notebook as-is.This is the one place SegNeXt needs more machinery than B0. `SegformerForSemanticSegmentation` canrebuild itself from a config because the class ships in `transformers`; here the class definitiononly exists in this project, so it is versioned alongside the weights rather than assumed. Section 15pushed it; this cell `exec`s it, which is why the architecture can never drift from the checkpoint.Inputs are raw full-resolution tiles; the same `INTER_AREA` downsample and the same logit upsample astraining are applied, so inference matches training preprocessing exactly.

In [ ]:
def load_from_hf(repo_id, token=None, device="cuda"):    import torch, numpy as np, cv2, types    import torch.nn.functional as F    from huggingface_hub import hf_hub_download    # the architecture travels with the weights -- see section 15    try:        src = open(hf_hub_download(repo_id, "segnext_model.py", token=token)).read()        mod = types.ModuleType("segnext_model")        exec(compile(src, "segnext_model.py", "exec"), mod.__dict__)    except Exception as e:        # repo predates the architecture-source push; fall back to the class in this        # kernel. Only valid because section 12 above defines the identical model.        if "SegNeXt" not in globals():            raise RuntimeError(                f"{repo_id} has no segnext_model.py and no SegNeXt class is defined here. "                f"Upload segnext_model.py to the repo, then retry.") from e        print(f"  NOTE: {repo_id} has no segnext_model.py -- using the in-kernel class. "              f"Upload it to the repo so this checkpoint stays loadable standalone.")        mod = types.SimpleNamespace(SegNeXt=globals()["SegNeXt"])    _p = hf_hub_download(repo_id, "best.pt", token=token)    try:                                              # torch>=2.6 weights_only default        ck = torch.load(_p, map_location=device, weights_only=True)    except Exception:        ck = torch.load(_p, map_location=device, weights_only=False)    m = mod.SegNeXt(ck["arch"]).to(device).eval()    m.load_state_dict(ck["model_state"])    return m, ck["cfg"], ck["metrics"]@torch.no_grad()def predict_paths(model, c, paths, device="cuda", batch=8):    import torch, numpy as np, cv2    import torch.nn.functional as F    mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)    out = []    for i in range(0, len(paths), batch):        xs = []        for p in paths[i:i + batch]:            im = cv2.imread(str(p), cv2.IMREAD_COLOR)            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))        x = torch.stack(xs).to(device)        with torch.cuda.amp.autocast():            lo = model(x)            lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)            out.append(lo.float().argmax(1).cpu().numpy())    return np.concatenate(out)# --- example ---INFER_REPO = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-block-fold0"inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)print(f"loaded {INFER_REPO} | val mIoU {inf_metrics['mIoU']:.4f}")demo = [IMG_DIR / r[2] for r in va_recs[:4]]preds = predict_paths(inf_model, inf_cfg, demo)print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## 23. Qualitative triptychs — image / GT / predictionThe palette is your BGR table reversed to RGB for matplotlib; ignore renders black. The selection isdeliberately not random: it shows the highest- and lowest-mIoU validation tiles, because the failurecases are what a reviewer wants to see. Watch for the two confusions you flagged — Unused Land vsAgricultural, and urban trees vs Forest.**The figure worth building from these.** You now have three models that fail on thin structures forthree different reasons, and a three-column comparison on the *same* worst-case tiles is a muchstronger paper figure than three separate qualitative panels:- **U-Net** fragments roads — high-resolution skip connections, no long-range context.- **B0** thickens or drops them — logits at 1/4 with self-attention smoothing.- **SegNeXt-T** should either be visibly better (the strip kernels tracking road runs across the  tile) or visibly *blockier* (logits at 1/8 quantising road edges to 8 px steps). Which of those you  see is the finding, and it is the direct evidence for or against the `FUSE_STAGE1` ablation.

In [ ]:
import matplotlib.pyplot as pltfrom matplotlib.patches import PatchPALETTE_RGB = np.zeros((256, 3), np.uint8)for bgr, idx in BGR_TO_CLASS.items():    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlibPALETTE_RGB[255] = (0, 0, 0)def colorize(lab): return PALETTE_RGB[lab]@torch.no_grad()def per_image_miou(recs, k=3):    scores = []    for j in range(0, len(recs), 16):        chunk = recs[j:j + 16]        xs, ys = zip(*[val_ds[val_ds.recs.index(r)] for r in chunk])        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()        with torch.cuda.amp.autocast(enabled=cfg.AMP):            p = forward_logits(best_model, x).float().argmax(1)        for b, r in enumerate(chunk):            c = ConfMat(cfg.N_CLASSES); c.update(p[b], y[b])            scores.append((metrics_from_cm(c.numpy(), cfg.CLASS_NAMES)["mIoU"], r))    scores.sort(key=lambda t: t[0])    return scores[-k:][::-1], scores[:k]best_imgs, worst_imgs = per_image_miou(va_recs, k=3)def triptych(items, title):    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))    axes = np.atleast_2d(axes)    for r, (score, rec) in enumerate(items):        _, _, iname, mname = rec        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)        gt = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        x, _ = val_ds[val_ds.recs.index(rec)]        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):            pr = forward_logits(best_model, x[None].cuda()).float().argmax(1)[0].cpu().numpy().astype(np.uint8)        pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)   # blank ignore for fair viewing        for c, (im, t) in enumerate([(img, iname), (colorize(gt), "ground truth"),                                     (colorize(pr_vis), f"prediction (mIoU {score:.3f})")]):            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]    handles.append(Patch(color="black", label="ignore/unannotated"))    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)    fig.suptitle(f"{title} - SegNeXt-T, {cfg.SPLIT_MODE} fold {cfg.FOLD}", fontsize=12)    plt.tight_layout(rect=[0, 0.04, 1, 0.97])    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()    hf_push(p, os.path.basename(p), "qualitative")triptych(best_imgs, "best cases")triptych(worst_imgs, "worst cases")

## 24. Confusion matrix figure

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])fig, ax = plt.subplots(figsize=(7.2, 6))im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)ax.set_xticks(range(6)); ax.set_yticks(range(6))ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")ax.set_title(f"Row-normalized confusion - SegNeXt-T, {cfg.SPLIT_MODE} fold {cfg.FOLD}")for i in range(6):    for j in range(6):        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,                color="white" if cmn[i, j] > 0.5 else "black")fig.colorbar(im, fraction=0.046)plt.tight_layout()p = f"{cfg.OUT_DIR}/confusion_{tag}.png"plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`. Internet is   needed for the MSCAN-T weights as well as pip/HF/W&B — without it, download   `mscan_t_20230227-119e8c9f.pth` separately, attach it as a Kaggle dataset, and set   `PRETRAINED_LOCAL`.2. Section 5 must print **channel-order check PASSED**.3. Point `PREBUILT_CACHE` at the 512 cache from the U-Net/B0 runs — it is byte-identical, and   section 6 becomes a no-op.4. **Section 12 must print all five of these.** Any one of them wrong means stop:   - `model source written and loaded: .../segnext_model.py (275 lines)`   - `params: 4.23 M`   - `pretrained: loaded N/N backbone tensors (100.0%)`   - `head fuses at stride 8 -> logits 64x64, upsampled 8x to 512`   - `logits after upsample: (1, 6, 512, 512)`5. Section 15 must print `architecture source verified in <repo>`. It re-queries the repo after   the upload, so this is proof the file is actually there — not just that the call returned.   Without it, `best.pt` cannot be rebuilt outside the kernel that trained it.6. Run the six fold-runs one per session, changing only `SPLIT_MODE` and `FOLD`:   `block/0, block/1, block/2, random/0, random/1, random/2`. **Do the three block folds first** —   see the quota arithmetic in section 18.7. Re-running the same fold after a timeout resumes automatically from `last.pt`.8. After all six, run section 21 for the aggregate table, the leakage gap, and the B0 thin-structure   comparison.### If the pretrained load failsThe checkpoint URL is the mmseg-converted MSCAN-T. The loader normalises the three naming variantsthat exist in the wild, but if a fourth appears it will raise with a printed sample of themissing/unused keys. **Paste that sample rather than setting `PRETRAINED_REQUIRED=False`.** Traininga 3.9 M-parameter backbone from scratch on 272 independent scenes will produce a number, and it willbe a meaningless one — a randomly-initialised row in a table where every other row isImageNet-pretrained is not a comparison of architectures.### Protocol fields — do not diverge from the B0 run`LABEL_SMOOTHING=0.05`, `COS_HORIZON_EPOCHS=120`, `USE_EMA=True`, `EN_BETA_AUTO=True`,`DETERMINISTIC=True`. These are set here to match B0. If you chose option (b) in the B0 checklist(smoothing off, horizon 200, EMA off, to match the original U-Net), **change them here too beforefold 0** — a table mixing the two is measuring the regularizer, not the architecture.`LR_DECODER=6e-4` is *not* in that category. It is a per-architecture value read off SegNeXt's ownADE20K config, on the same footing as UNetFormer's 6e-4 and B0's 6e-5. Say so in the methods section.### If it underperforms, in this order1. **Check Road/River specifically before anything else.** If overall mIoU is competitive but Road   and River lag B0, the 1/8 fusion stride is the suspect, and `FUSE_STAGE1=True` is the ablation   that tests it directly — one config field, one extra row in the table, and a genuinely   interesting result either way. This is the most likely outcome worth chasing.2. If it overfits the way the U-Net did (train loss under 0.1 while val mIoU is flat for 15+ epochs),   raise `DROP_PATH_RATE` to 0.2. Per-architecture, costs no re-runs.3. If grad norms sit pinned at the 1.0 clip past the warmup, drop `LR_DECODER` to 3e-4 and re-run   every SegNeXt fold at the new value.Do not touch augmentation. If you do, it changes for all seven baselines.### For the paperSegNeXt is Apache-2.0 (Visual-Attention-Network, Tsinghua). Cite Guo et al., *SegNeXt: RethinkingConvolutional Attention Design for Semantic Segmentation*, NeurIPS 2022, and Geng et al., *IsAttention Better Than Matrix Decomposition?*, ICLR 2021, for the Hamburger decoder. When you describethe architecture family in the comparison table, "convolutional attention" is the term the authorsuse — it is neither a plain CNN nor a transformer, and that distinction is precisely why this rowearns its place next to U-Net, UNetFormer, and SegFormer-B0.